# Alpha-Mind Quick Start 3: Portfolio Construction

The alpha-mind portfolio package provides portfolio-construction utilities.
- Different construction methods correspond to different optimization problems.
- The package includes linear and mean-variance construction methods.

### Linear Construction: *linear_builder*
- Objective: maximize expected portfolio return subject to linear constraints.
- Linear constraints include:
  - Lower and upper asset-weight bounds *lbound, ubound*.
  - Lower and upper portfolio risk-exposure bounds *risk_target*: the function multiplies the factor matrix by the weights to obtain portfolio risk exposure.
  - The turnover limit for position adjustments *turn_over_target*, where turnover is the L1 norm of the difference between target and current weight vectors, namely the sum of absolute differences.

In [ ]:
# Assume five assets
# Current asset weights
# Expected asset-return vector
# Asset risk-factor matrix with one risk factor
# Constraints
# Constraint 1 - lower and upper asset-weight bounds
# Constraint 2 - lower and upper portfolio risk-exposure bounds
# Constraint 3 - turnover upper bound for position adjustments (the lower bound is zero and need not be set)
import numpy as np
import pyarrow as pa
from alphamind.portfolio.constraints import BoundaryType, LinearConstraints, create_box_bounds


### Mean-Variance Construction: *mean_variance_builder*
- Build a model from the classic mean-variance portfolio-optimization problem.
- The objective function is:


\begin{align*}
&\mathop{\arg\max}_{\omega} \ \  \omega_a R - \frac{1}{2} \lambda \sigma   \\
&s.t.\quad
\begin{cases}
\omega_L  \le \omega_a \le \omega_U  \\
\sigma_L \le \sigma \le \sigma_U
\end{cases}
\end{align*}


where
- $\omega_a$ is the vector of active portfolio weights. If $\omega$ and $\omega_b$ denote absolute portfolio and benchmark asset-weight vectors, then $\omega_a =\omega -\omega_b$.
- $R$ is the vector of expected asset returns.
- $\lambda$ is the risk-aversion coefficient, and $\sigma$ is the variance of total portfolio returns.

##### Function Parameters and Usage

*mean_variance_builder* accepts:
- Expected asset-return vector *er*.
- Risk model *risk_model* (optional).
- Benchmark asset-weight vector *bm*.
- Lower and upper asset-weight bounds *lbound, ubound*.
- Portfolio risk-exposure matrix *risk_exposure* (optional).
- Lower and upper portfolio risk-exposure bounds *risk_target* (optional).
- Risk-aversion coefficient *lam* (optional).

NOTE 1: The lower and upper asset-weight bounds *lbound, ubound* apply to the absolute weight vector.
- The code calculates active-weight bounds: *$\omega_L$ = lbound - $\omega_b$* and *$\omega_U$ = ubound - $\omega_b$*. It finally adds $\omega_b$ back to the optimizer's solution to obtain optimal absolute portfolio weights.
- The code also calculates active-return risk bounds: *risk_target_lbound - risk_exposure $\omega_b$* and *risk_target_ubound + risk_exposure $\omega_b$*.

NOTE 2: To calculate portfolio variance, *mean_variance_builder* accepts two input forms:
- The asset-return covariance matrix $\Sigma$, giving

\begin{equation*}
    \sigma = \omega_a^T\Sigma \omega_a
\end{equation*}

where $\Sigma$ corresponds to risk_model['cov'].

- A risk model comprising the factor-return covariance matrix $F$ and the idiosyncratic-return covariance matrix $\Delta$ (a diagonal matrix):
\begin{equation*}
    \sigma = \omega_a^T \left(X F X^T + \Delta \right)  \omega_a
\end{equation*}

Because there are generally fewer risk factors than stocks, $\omega_a^T X F X^T \omega_a$ has lower computational complexity than $\omega_a^T\Sigma \omega_a$, making the second method more efficient.

Here, $F$ corresponds to risk_model['factor_cov'], $\Delta$ to risk_model['idsync'], and $X$ to risk_model['factor_loading'].

In [ ]:
# Assume three assets
# Expected asset-return vector
# Return covariance matrix
# Idiosyncratic return variance (idiosyncratic risk)
# The resulting covariance matrix is
# Asset risk-exposure matrix with two risk factors
# Benchmark index weight vector
# Constraints
# Constraint 1 - lower and upper absolute asset-weight bounds
# Constraint 2 - lower and upper portfolio risk-exposure bounds
# Build a risk model using the asset covariance matrix directly to calculate portfolio variance
from alphamind.portfolio.meanvariancebuilder import mean_variance_builder

er = np.array([0.01, 0.02, 0.03, 0.015])


In [ ]:
benchmark = np.full(4, 0.25)
bounds = create_box_bounds(["total"], BoundaryType.ABSOLUTE, 0.0, 0.0)
constraints = LinearConstraints(bounds, pa.table({"total": np.ones(4)}), benchmark)


In [ ]:
risk_model = {"cov": np.eye(4) * 0.02, "factor_cov": None, "factor_loading": None, "idsync": None}
status, objective, weights = mean_variance_builder(
    er,
    risk_model,
    benchmark,
    0.0,
    0.5,
    constraints.risk_exp,
    constraints.risk_targets(),
)


In [ ]:
assert status in ("optimal", "optimal_inaccurate")
np.testing.assert_allclose(weights.sum(), 1.0)
weights
